# OSM BaseData — raw inputs only, no model output

*Notebook style follows Charlotte Ellison's existing RideScore DC notebooks.
The OSM/DC join methodology was developed jointly by Charlotte and EChO Ory.
This notebook was drafted with Claude (Anthropic), editing and structure by
EChO.*

**Just want the data, not the analysis?** Grab `osm_basedata_raw.geojson` or
`osm_basedata_raw.csv` directly from this folder — a small, committed extract,
no notebook required. Keep reading if you want to check the data's quality
before building a model on it, or if you need a *fresher* pull than the
committed extract.

**What this is:** raw street segment attributes, from OpenStreetMap and
(optionally) DC Open Data, with **no model output of any kind** — no LTS
score, no BNA stress score, no RideScore. Every model in this project should
be able to start from these same raw columns.

**Why two ways to get data, not one file:** a small extract is committed to
git so this notebook works with no network and no wait. A *full-DC* pull is
never committed — at ~100k segments it would blow past GitHub's file-size
limits — so it only ever exists locally, produced on demand by the fetch cell
below. `SUBSET` picks between the two.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely import wkt
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium import GeoJson
from IPython.display import display
import osmnx as ox

## Parameters

In [2]:
SUBSET = 1
# 1 = small committed extract (osm_basedata_raw.geojson) -- fast, has the DC join
# 0 = full DC, pulled live from OSM just now -- slow, OSM only, no DC join yet
#     (the DC-side conflation hasn't been run at full-DC scale -- see the
#     open questions at the end)

SUBSET_BBOX = (-77.0445601, 38.9097148, -77.0054513, 38.9287761)  # WGS84, only used when SUBSET=1

INCLUDE_DC_COMPARISON = True  # only has an effect when SUBSET=1

DATA_DIR = "."
FULL_DC_CACHE = "cache/osm_full_dc.geojson"  # gitignored -- see below, never commit this file

## Getting the data

`SUBSET=1` reads the small committed file. `SUBSET=0` fetches live from OSM —
this actually hits the network and can take a while for all of DC; the
`graph_from_place`/`graph_to_gdfs` pattern below is confirmed working (tested
against a single DC neighborhood), but **a full-city run has not been timed or
tested end-to-end** — budget real time for it, don't run it for the first time
five minutes before you need the result.

If you run this, add `cache/` to `.gitignore` before committing anything else
in this folder — the output is too large for git.

In [3]:
RAW_OSM_TAGS = [
    "highway", "name", "access", "bicycle", "oneway", "oneway:bicycle", "maxspeed",
    "lanes", "lanes:forward", "lanes:backward",
    "cycleway", "cycleway:left", "cycleway:left:buffer", "cycleway:left:oneway", "cycleway:left:width",
    "cycleway:right", "cycleway:right:buffer", "cycleway:right:oneway", "cycleway:right:width",
    "cycleway:both", "cycleway:both:buffer", "cycleway:both:width", "cycleway:buffer", "cycleway:width",
    "parking:left", "parking:right", "parking:both", "parking:lane:left", "parking:lane:right", "parking:lane:both",
    "parking:left:restriction", "parking:right:restriction", "parking:both:restriction",
    "tracktype", "turn:lanes", "turn:lanes:forward", "turn:lanes:backward", "width", "footway",
]


def fetch_osm(place="Washington, District of Columbia, USA"):
    """Pull a street network directly from OSM with the raw tags this project needs.

    Confirmed working against a single neighborhood place-name. Not yet timed
    or run against all of DC -- do that deliberately, not as a last-minute
    check before a deadline.
    """
    ox.settings.useful_tags_way = list(set(ox.settings.useful_tags_way) | set(RAW_OSM_TAGS))
    graph = ox.graph_from_place(place, network_type="all", simplify=False)
    _, edges = ox.graph_to_gdfs(graph)
    edges = edges.reset_index().rename(columns={"osmid": "osm_id"})
    keep = ["osm_id"] + [t for t in RAW_OSM_TAGS if t in edges.columns] + ["geometry"]
    return edges[keep].to_crs("EPSG:4326")


if SUBSET:
    segments = gpd.read_file(f"{DATA_DIR}/osm_basedata_raw.geojson")
    minx, miny, maxx, maxy = SUBSET_BBOX
    segments = segments.cx[minx:maxx, miny:maxy]
    if not INCLUDE_DC_COMPARISON:
        segments = segments.drop(columns=[c for c in segments.columns if c.startswith("dc_")])
else:
    import os
    os.makedirs("cache", exist_ok=True)
    segments = fetch_osm()  # OSM only -- no dc_* columns at full-DC scale yet
    segments.to_file(FULL_DC_CACHE, driver="GeoJSON")

print(f"{len(segments)} segments  (SUBSET={SUBSET})")
segments.head(3)

3760 segments  (SUBSET=1)


,osm_id,osm_highway,osm_name,osm_access,osm_bicycle,osm_oneway,osm_oneway_bicycle,osm_maxspeed,osm_lanes,osm_lanes_forward,...,dc_RIGHTTURN_CURBLANE_EXCL,dc_RIGHTTURN_CURBLANE_EXCL_LEN,dc_LEFTTURN_EXCLUSIVE,dc_RIGHTTURN_EXCLUSIVE,dc_BUSLANE_INBOUND,dc_BUSLANE_OUTBOUND,dc_AADT,dc_AADT_YEAR,match_confidence,geometry
0,5974756,trunk,Georgia Avenue Northwest,NaN,NaN,no,NaN,30 mph,4,NaN,...,None,0.0,None,None,NaN,NaN,13028.0,2020.0,1.0,"LINESTRING (-77.02212 38.92133, -77.02215 38.9..."
1,5974756,trunk,Georgia Avenue Northwest,NaN,NaN,no,NaN,30 mph,4,NaN,...,None,0.0,None,None,NaN,NaN,13028.0,2020.0,1.0,"LINESTRING (-77.02217 38.92173, -77.02225 38.9..."
2,5974756,trunk,Georgia Avenue Northwest,NaN,NaN,no,NaN,30 mph,4,NaN,...,None,0.0,None,None,NaN,NaN,13028.0,2020.0,1.0,"LINESTRING (-77.02225 38.92228, -77.02226 38.9..."


## What's in here

- `osm_id` — segment identity going forward
- `osm_*` — straight from OpenStreetMap tags
- `dc_*` — straight from DC Open Data's DDOT roadway blocks, **only present
  when `SUBSET=1`** — the full-DC join hasn't been run yet
- `dc_blockkey` / `match_confidence` — DDOT's identity and the join's overlap
  confidence, both from the `SUBSET=1` extract only

In [4]:
for c in segments.columns:
    print(c)

osm_id
osm_highway
osm_name
osm_access
osm_bicycle
osm_oneway
osm_oneway_bicycle
osm_maxspeed
osm_lanes
osm_lanes_forward
osm_lanes_backward
osm_cycleway
osm_cycleway_left
osm_cycleway_left_buffer
osm_cycleway_left_oneway
osm_cycleway_left_width
osm_cycleway_right
osm_cycleway_right_buffer
osm_cycleway_right_oneway
osm_cycleway_right_width
osm_cycleway_both
osm_cycleway_both_buffer
osm_cycleway_both_width
osm_cycleway_buffer
osm_cycleway_width
osm_parking_left
osm_parking_right
osm_parking_both
osm_parking_lane_left
osm_parking_lane_right
osm_parking_lane_both
osm_parking_left_restriction
osm_parking_right_restriction
osm_parking_both_restriction
osm_tracktype
osm_turn_lanes
osm_turn_lanes_forward
osm_turn_lanes_backward
osm_width
osm_footway
dc_blockkey
dc_ROUTEID
dc_ROUTENAME
dc_ROADTYPE
dc_STREETNAME
dc_STREETTYPE
dc_DCFUNCTIONALCLASS
dc_FHWAFUNCTIONALCLASS
dc_TOTALTRAVELLANES
dc_TOTALTRAVELLANESINBOUND
dc_TOTALTRAVELLANESOUTBOUND
dc_TOTALTRAVELLANESBIDIRECTIONAL
dc_TOTALTRAVELLANES

## Match quality (SUBSET=1 only)

In [5]:
if "dc_blockkey" in segments.columns:
    has_dc = segments["dc_blockkey"].notna()
    good = has_dc & (segments["match_confidence"] >= 0.8)
    weak = has_dc & (segments["match_confidence"] < 0.8)
    none_ = ~has_dc
    print(f"good match  (>= 0.8):  {good.sum():>5}  {good.sum()/len(segments):.1%}")
    print(f"weak match  (<  0.8):  {weak.sum():>5}  {weak.sum()/len(segments):.1%}")
    print(f"no DC match at all:    {none_.sum():>5}  {none_.sum()/len(segments):.1%}")
else:
    print("No DC join at this scale -- run with SUBSET=1 to see match quality.")

good match  (>= 0.8):   3387  90.1%
weak match  (<  0.8):    168  4.5%
no DC match at all:      205  5.5%


## Completeness check: OSM vs DC, where both exist (SUBSET=1 only)

This is the concrete case for keeping DC alongside OSM: DC's raw attributes
are meaningfully more complete in DC than OSM's own tagging is, for several
of these fields.

In [6]:
pairs = [
    ("osm_maxspeed", "dc_SPEEDLIMITS_OB"),
    ("osm_lanes", "dc_TOTALTRAVELLANES"),
    ("osm_cycleway_right", "dc_BIKELANE_CONVENTIONAL"),
    ("osm_width", "dc_TOTALCROSSSECTIONWIDTH"),
]

for osm_col, dc_col in pairs:
    if osm_col in segments.columns and dc_col in segments.columns:
        osm_pct = segments[osm_col].notna().mean()
        dc_pct = segments[dc_col].notna().mean()
        print(f"{osm_col:<22} populated {osm_pct:>6.1%}   |   {dc_col:<28} populated {dc_pct:>6.1%}")

osm_maxspeed           populated  60.9%   |   dc_SPEEDLIMITS_OB            populated  74.3%
osm_lanes              populated  62.3%   |   dc_TOTALTRAVELLANES          populated  94.5%
osm_cycleway_right     populated  19.8%   |   dc_BIKELANE_CONVENTIONAL     populated  21.8%
osm_width              populated   1.0%   |   dc_TOTALCROSSSECTIONWIDTH    populated  94.5%


## Where the weak and missing matches are (SUBSET=1 only)

In [7]:
if "dc_blockkey" in segments.columns:
    segments_wgs84 = segments.to_crs("EPSG:4326") if segments.crs and segments.crs.to_epsg() != 4326 else segments

    m = folium.Map(location=[38.918, -77.025], zoom_start=15, tiles="CartoDB positron")

    GeoJson(segments_wgs84[good][["geometry"]].__geo_interface__, name="good match (>= 0.8)",
            style_function=lambda x: {"color": "#2166ac", "weight": 3, "opacity": 0.8}).add_to(m)
    GeoJson(segments_wgs84[weak][["geometry"]].__geo_interface__, name="weak match (< 0.8)",
            style_function=lambda x: {"color": "#f46d43", "weight": 4, "opacity": 0.9}).add_to(m)
    GeoJson(segments_wgs84[none_][["geometry"]].__geo_interface__, name="no DC match",
            style_function=lambda x: {"color": "#999999", "weight": 3, "opacity": 0.9}).add_to(m)

    folium.LayerControl().add_to(m)
    display(m)
else:
    print("No DC join at this scale -- nothing to map here yet.")

/usr/local/lib/python3.12/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


## Open questions for the group

1. When OSM and DC attributes disagree, which one wins, per attribute?
2. What happens to the ~4-5% weak-match segments -- drop, flag, or trust as-is?
3. What happens to segments with no DC match at all?
4. **The DC join has only been run on this small subset.** Running the same
   buffer + bearing conflation at full-DC scale is real, untested work --
   don't assume `SUBSET=0` gives you `dc_*` columns until that's done.
5. **BNA needs more than this file has**: destination data (schools, jobs,
   parks, retail) and census population, for the connectivity side of BNA --
   a separate data source, not a column added here.
6. Full-DC live fetch timing/reliability is unverified -- test it with real
   time budget before depending on it.